In [0]:
file_name = dbutils.widgets.text("file_name","")

In [0]:
file_name = dbutils.widgets.get("file_name")

In [0]:
file_name

In [0]:
Fact_df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
        .option("inferSchema","True") \
            .option("cloudFiles.schemaLocation",f"abfss://silver@spotifystorageaccount02.dfs.core.windows.net/{file_name}/check_point") \
                .load(f"abfss://bronze@spotifystorageaccount02.dfs.core.windows.net/{file_name}")


In [0]:
display(Fact_df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


import os 
import sys


##project_pth = os.path.join(os.getcwd(),'..','..')
##sys.path.append(project_pth)


utils_path = "/Workspace/Users/syammangat2001@gmail.com/SpotifyAzureProject/Spotify_DAB_bundles/utils"


if utils_path not in sys.path:
    sys.path.append(utils_path)

    
print(sys.path)

In [0]:
import sys

utils_path = "/Workspace/Users/syammangat2001@gmail.com/Spotify_end_to_end_data_pipeline/Databricks/SpotifyAzureProject/Spotify_DAB_bundles/src/Spotify_DAB_bundles/utils/"

if utils_path not in sys.path:
    sys.path.append(utils_path)



In [0]:
print(sys.path)

In [0]:
from Transformations import reusable

Fact_df  = reusable().dropColumns(Fact_df,['_rescued_data'])


In [0]:
Fact_df.writeStream.format("delta") \
    .outputMode("append") \
        .option("checkpointLocation",f"abfss://silver@spotifystorageaccount02.dfs.core.windows.net/{file_name}/check_point") \
            .trigger(once=True) \
                .option("path",f"abfss://silver@spotifystorageaccount02.dfs.core.windows.net/{file_name}/data") \
                    .toTable(f"dev_catalog.silver.{file_name}")
